<a href="https://colab.research.google.com/github/SJH-official/Application_Developments/blob/main/%EC%95%B1%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D_%EA%B8%B0%EB%A7%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚴 AdventureWorks Sales Analysis
## CRM 관점의 고객 구매 예측 — Random Forest Classifier & Regressor

---
| 항목 | 내용 |
|------|------|
| 데이터 | AdventureWorks Sales (FY2018 ~ FY2020) |
| 분석 목적 | 전체 현황 파악 → EDA → CRM 기반 구매 예측 |
| 예측 모델 | Random Forest Classifier (Buy/Not Buy) + Regressor (판매량) |

---
### 📌 분석 흐름
```
1. 라이브러리 & 데이터 로드
2. 데이터 전처리  ← 결측치 / 이상치 시각화 & 제거 / 중복 검사
3. 기본 통계분석  ← describe / 분포 / 상관관계
4. EDA            ← 시간·공간 / 고객·상품 / 특이현상 발굴
5. 예측 모델      ← RF Classifier + Regressor + 인터랙티브 시뮬레이션
6. CRM 인사이트
```

## 1️⃣ 라이브러리 & 데이터 로드

In [1]:
# ── 라이브러리 ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, accuracy_score,
    mean_absolute_error, r2_score, ConfusionMatrixDisplay
)

# ── 시각화 스타일 ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0A1628',
    'axes.facecolor':   '#0D2240',
    'axes.edgecolor':   '#1E3A5F',
    'axes.labelcolor':  '#94A3B8',
    'xtick.color':      '#94A3B8',
    'ytick.color':      '#94A3B8',
    'text.color':       '#FFFFFF',
    'grid.color':       '#1E3A5F',
    'grid.linestyle':   '--',
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   13,
    'axes.titlecolor':  '#FFFFFF',
})
PALETTE = ['#00C8FF','#4AEDC4','#FFD166','#FF6B6B','#9B59B6','#27AE60','#E67E22']

print('✅ 라이브러리 로드 완료')

✅ 라이브러리 로드 완료


In [ ]:
# ── 파일 업로드 (Colab) ─────────────────────────────────────────────────────
from google.colab import files
print('📂 AdventureWorks_Sales.xlsx 파일을 업로드해주세요')
uploaded = files.upload()
FNAME = list(uploaded.keys())[0]
print(f'✅ 업로드 완료: {FNAME}')

📂 AdventureWorks_Sales.xlsx 파일을 업로드해주세요


In [ ]:
# ── 7개 시트 로드 ────────────────────────────────────────────────────────────
sales     = pd.read_excel(FNAME, sheet_name='Sales_data')
customers = pd.read_excel(FNAME, sheet_name='Customer_data')
products  = pd.read_excel(FNAME, sheet_name='Product_data')
dates     = pd.read_excel(FNAME, sheet_name='Date_data')
territory = pd.read_excel(FNAME, sheet_name='Sales Territory_data')
reseller  = pd.read_excel(FNAME, sheet_name='Reseller_data')
order     = pd.read_excel(FNAME, sheet_name='Sales Order_data')

tables = {'Sales':sales,'Customers':customers,'Products':products,
          'Dates':dates,'Territory':territory,'Reseller':reseller,'Order':order}

print(f"{'테이블':<14} {'Rows':>8} {'Cols':>6}")
print('-' * 32)
for name, t in tables.items():
    print(f"{name:<14} {len(t):>8,} {len(t.columns):>6}")

## 2️⃣ 데이터 전처리
### 2-1. 테이블 병합

In [ ]:
# ── 테이블 병합 ──────────────────────────────────────────────────────────────
df = (
    sales
    .merge(order[['SalesOrderLineKey','Channel']], on='SalesOrderLineKey', how='left')
    .merge(customers.rename(columns={
               'City':'CustCity', 'State-Province':'CustState',
               'Country-Region':'CustCountry'}), on='CustomerKey', how='left')
    .merge(products[['ProductKey','Category','Subcategory',
                      'Product','Model','Standard Cost','List Price']],
           on='ProductKey', how='left')
    .merge(dates[['DateKey','Fiscal Year','Fiscal Quarter','Month']],
           left_on='OrderDateKey', right_on='DateKey', how='left')
    .merge(territory, on='SalesTerritoryKey', how='left')
)

# 파생변수: 월 번호 (DateKey = YYYYMMDD)
df['month_num'] = (df['OrderDateKey'] % 10000) // 100

print(f'병합 완료 → {df.shape[0]:,} rows × {df.shape[1]} cols')
df.head(3)

### 2-2. 결측치 확인 및 제거

In [ ]:
# ── 결측치 현황 시각화 ───────────────────────────────────────────────────────
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print(f'결측치가 있는 컬럼: {len(missing)}개\n')
print(f"{'컬럼':<30} {'건수':>7} {'비율':>7}  막대")
print('-' * 60)
for col, cnt in missing.items():
    pct = cnt / len(df) * 100
    bar = '█' * int(pct / 2) + '░' * (10 - int(pct / 2))
    print(f"{col:<30} {cnt:>7,} {pct:>6.1f}%  {bar}")

# 시각화
fig, ax = plt.subplots(figsize=(9, 3))
fig.patch.set_facecolor('#0A1628')
missing.plot(kind='bar', ax=ax, color=PALETTE[3], edgecolor='none')
ax.set_title('컬럼별 결측치 건수', pad=10)
ax.set_ylabel('결측 건수')
ax.set_xlabel('')
for bar in ax.patches:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{int(bar.get_height()):,}', ha='center', fontsize=9, color='white')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

### 2-3. 이상치 확인 및 제거 (IQR 방법)

In [ ]:
# ── 이상치 시각화 (제거 전) ──────────────────────────────────────────────────
outlier_cols = ['Sales Amount', 'Unit Price', 'Order Quantity']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.patch.set_facecolor('#0A1628')
fig.suptitle('이상치 확인 — Boxplot (제거 전)', fontsize=13,
             fontweight='bold', color='white', y=1.02)

for ax, col, color in zip(axes, outlier_cols, PALETTE):
    bp = ax.boxplot(df[col].dropna(), patch_artist=True, notch=False,
                    medianprops=dict(color='white', linewidth=2),
                    boxprops=dict(facecolor=color, alpha=0.6),
                    whiskerprops=dict(color='#94A3B8'),
                    capprops=dict(color='#94A3B8'),
                    flierprops=dict(marker='o', color=PALETTE[3],
                                   alpha=0.3, markersize=3))
    ax.set_title(col, fontsize=11)
    ax.set_ylabel('값')
    # IQR 이상치 수 표시
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    ax.set_xlabel(f'이상치 {n_out:,}건 ({n_out/len(df)*100:.1f}%)',
                  color=PALETTE[3], fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── 이상치 & 결측치 제거 ──────────────────────────────────────────────────────
before = len(df)

# ① Sales Amount ≤ 0 (환불·오류 데이터)
df = df[df['Sales Amount'] > 0]

# ② 핵심 컬럼 결측 제거
df = df.dropna(subset=['Category', 'Channel', 'Fiscal Year'])

# ③ ShipDateKey 컬럼 제거 (배송 미완료 = 정상, 분석 불필요)
df = df.drop(columns=['ShipDateKey'], errors='ignore')

after = len(df)
internet_df = df[df['CustomerKey'] > 0].copy()   # 인터넷 채널 (유효 고객)

print('[ 전처리 결과 ]')
print(f'  전처리 전 : {before:>8,} 행')
print(f'  전처리 후 : {after:>8,} 행  (제거 {before-after:,}건)')
print(f'  인터넷 채널: {len(internet_df):>7,} 건')
print(f'  Reseller  : {len(df[df["Channel"]=="Reseller"]):>7,} 건')

### 2-4. 중복 데이터 검사

In [ ]:
# ── 중복 검사 ────────────────────────────────────────────────────────────────
dup_count = df.duplicated(subset='SalesOrderLineKey').sum()
print(f'SalesOrderLineKey 기준 중복 행: {dup_count}건')
print(f'→ {"중복 없음 ✅" if dup_count == 0 else f"중복 {dup_count}건 제거 필요 ⚠️"}')

## 3️⃣ 기본 통계 분석 (Descriptive Statistics)

In [ ]:
# ── 3-1. 수치형 변수 기술통계 ───────────────────────────────────────────────
NUM_COLS = ['Sales Amount','Order Quantity','Unit Price',
            'Product Standard Cost','Total Product Cost']

desc = df[NUM_COLS].describe().T
desc['cv(%)'] = (desc['std'] / desc['mean'] * 100).round(1)  # 변동계수
print('[ 기술통계량 ]')
print(desc[['count','mean','std','min','25%','50%','75%','max','cv(%)']].round(2).to_string())

In [ ]:
# ── 3-2. 수치형 변수 분포 (히스토그램) ──────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.patch.set_facecolor('#0A1628')
fig.suptitle('수치형 변수 분포', fontsize=14, fontweight='bold', color='white')
axes = axes.flatten()

for i, (col, color) in enumerate(zip(NUM_COLS, PALETTE)):
    data = df[col].dropna()
    axes[i].hist(data, bins=50, color=color, alpha=0.8, edgecolor='none')
    axes[i].axvline(data.mean(),   color='white',    linestyle='--', linewidth=1.5, label=f'평균 {data.mean():.0f}')
    axes[i].axvline(data.median(), color=PALETTE[3], linestyle=':',  linewidth=1.5, label=f'중앙값 {data.median():.0f}')
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel('값')
    axes[i].set_ylabel('빈도')
    axes[i].legend(fontsize=8)

axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3-3. 상관관계 히트맵 ────────────────────────────────────────────────────
corr = df[NUM_COLS].corr()

fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor('#0A1628')
ax.set_facecolor('#0A1628')

mask = np.triu(np.ones_like(corr, dtype=bool))   # 상삼각 마스킹
sns.heatmap(corr, ax=ax, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            linewidths=0.5, linecolor='#0A1628',
            annot_kws={'size': 11, 'weight': 'bold'},
            cbar_kws={'shrink': 0.8})
ax.set_title('변수 간 상관관계 히트맵', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

print('\n📌 주요 상관관계 해석')
print('  Sales Amount ↔ Total Product Cost : r =', round(corr.loc['Sales Amount','Total Product Cost'], 2), '(매우 강한 양의 상관)')
print('  Sales Amount ↔ Unit Price         : r =', round(corr.loc['Sales Amount','Unit Price'], 2), '(강한 양의 상관)')
print('  Unit Price   ↔ Product Std Cost   : r =', round(corr.loc['Unit Price','Product Standard Cost'], 2), '(매우 강한 양의 상관)')

In [ ]:
# ── 3-4. 범주형 변수 분포 ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.patch.set_facecolor('#0A1628')
fig.suptitle('범주형 변수 분포', fontsize=13, fontweight='bold', color='white')

cat_vars = [('Category','카테고리'), ('Channel','채널'), ('Fiscal Year','회계연도')]

for ax, (col, label), color in zip(axes, cat_vars, PALETTE):
    vc = df[col].value_counts()
    bars = ax.bar(vc.index, vc.values, color=color, edgecolor='none', width=0.5)
    for bar, val in zip(bars, vc.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                f'{val:,}', ha='center', fontsize=9, color='white')
    ax.set_title(label, fontsize=11)
    ax.set_ylabel('거래 건수')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
    plt.setp(ax.get_xticklabels(), rotation=20, ha='right')

plt.tight_layout()
plt.show()

## 4️⃣ EDA (탐색적 데이터 분석)
### 4-1. 전체 KPI 현황

In [ ]:
# ── KPI Dashboard ────────────────────────────────────────────────────────────
total_sales  = df['Sales Amount'].sum()
inet_sales   = internet_df['Sales Amount'].sum()
n_customers  = internet_df['CustomerKey'].nunique()
avg_per_cust = internet_df.groupby('CustomerKey')['Sales Amount'].sum().mean()
top10_share  = (
    internet_df.groupby('CustomerKey')['Sales Amount'].sum()
    .nlargest(int(n_customers * 0.1)).sum() / inet_sales * 100
)

print('=' * 50)
print('      📊  AdventureWorks KPI Dashboard')
print('=' * 50)
kpis = [
    ('💰 전체 총 매출',         f'${total_sales/1e6:.1f}M'),
    ('🌐 인터넷 채널 매출',     f'${inet_sales/1e6:.1f}M'),
    ('🏪 Reseller 채널 매출',   f'${(total_sales-inet_sales)/1e6:.1f}M'),
    ('👥 인터넷 고객 수',       f'{n_customers:,}명'),
    ('💳 고객당 평균 구매액',   f'${avg_per_cust:,.0f}'),
    ('🏆 상위 10% 고객 매출 비중', f'{top10_share:.1f}%'),
]
for label, val in kpis:
    print(f'  {label:<22}  {val:>10}')
print('=' * 50)

### 4-2. 시간 분석 — 연도별 · 분기별 · 월별

In [ ]:
# ── 연도별 매출 추이 + 분기별 추이 ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0A1628')

# 연도별
fy = internet_df.groupby('Fiscal Year')['Sales Amount'].sum().div(1e6)
bars = axes[0].bar(fy.index, fy.values, color=PALETTE[:3], width=0.5, edgecolor='none')
for bar, val in zip(bars, fy.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                 f'${val:.1f}M', ha='center', fontsize=11,
                 fontweight='bold', color='white')
axes[0].set_title('연도별 인터넷 채널 매출')
axes[0].set_ylabel('매출 ($M)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x:.0f}M'))
axes[0].grid(axis='y', alpha=0.4)
axes[0].set_axisbelow(True)

# 분기별 추이 (연도 × 분기)
fq = (internet_df.groupby(['Fiscal Year','Fiscal Quarter'])['Sales Amount']
      .sum().div(1e6).unstack())
for i, (fy_label, row) in enumerate(fq.iterrows()):
    axes[1].plot(row.index, row.values, marker='o', linewidth=2,
                 color=PALETTE[i], label=fy_label)
    for q, v in zip(row.index, row.values):
        axes[1].annotate(f'${v:.1f}M', (q, v), textcoords='offset points',
                         xytext=(0,8), ha='center', fontsize=8, color=PALETTE[i])
axes[1].set_title('분기별 인터넷 채널 매출 추이')
axes[1].set_ylabel('매출 ($M)')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)
axes[1].set_axisbelow(True)

plt.suptitle('시간 분석 — 연도 · 분기별 매출', fontsize=14,
             fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 월별 매출 히트맵 ─────────────────────────────────────────────────────────
pivot = (internet_df.groupby(['Fiscal Year','month_num'])['Sales Amount']
         .sum().div(1e3).unstack(level=0))
pivot.index.name = '월'

fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor('#0A1628')
ax.set_facecolor('#0A1628')

sns.heatmap(pivot, ax=ax, cmap='YlOrRd', linewidths=0.4,
            linecolor='#0A1628', annot=True, fmt='.0f',
            annot_kws={'size':9, 'color':'#0A1628'},
            cbar_kws={'label':'매출 ($K)', 'shrink':0.8})
ax.set_title('월별 × 연도별 인터넷 채널 매출 히트맵 ($K)', fontsize=13, pad=12)
ax.set_xlabel('회계연도')
ax.set_ylabel('월')
plt.tight_layout()
plt.show()

best_month = pivot.sum(axis=1).idxmax()
print(f'📌 특이현상: 연중 매출 최고 월 → {best_month}월  (FY2020 급성장이 주도)')

### 4-3. 공간 분석 — 국가별 · 지역별

In [ ]:
# ── 국가별 매출 + 지역별 채널 비교 ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0A1628')

# 국가별
country_s = (internet_df.groupby('CustCountry')['Sales Amount']
             .sum().div(1e6).sort_values(ascending=True))
colors_c = [PALETTE[0] if v == country_s.max() else PALETTE[2] for v in country_s]
axes[0].barh(country_s.index, country_s.values, color=colors_c, edgecolor='none', height=0.6)
for val, y in zip(country_s.values, range(len(country_s))):
    axes[0].text(val+0.1, y, f'${val:.1f}M', va='center', fontsize=10,
                 fontweight='bold', color='white')
axes[0].set_title('국가별 인터넷 채널 매출')
axes[0].set_xlabel('매출 ($M)')
axes[0].grid(axis='x', alpha=0.3)

# 지역별 × 채널
reg_ch = (df.groupby(['Region','Channel'])['Sales Amount']
           .sum().div(1e6).unstack().fillna(0)
           .sort_values('Reseller' if 'Reseller' in df['Channel'].values else df['Channel'].unique()[0], ascending=False))
x = np.arange(len(reg_ch)); w = 0.4
for i, (ch, color) in enumerate(zip(reg_ch.columns, [PALETTE[0], PALETTE[2]])):
    bars = axes[1].bar(x + (i-0.5)*w, reg_ch[ch], width=w,
                       color=color, label=ch, edgecolor='none')
axes[1].set_xticks(x)
axes[1].set_xticklabels(reg_ch.index, rotation=30, ha='right', fontsize=9)
axes[1].set_title('지역별 × 채널별 매출')
axes[1].set_ylabel('매출 ($M)')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_axisbelow(True)

plt.suptitle('공간 분석 — 국가 · 지역별 매출', fontsize=14,
             fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.show()

top2 = country_s.nlargest(2)
print(f'📌 특이현상: {top2.index[0]}·{top2.index[1]} 양강 구도 → 합산 {top2.sum()/internet_df["Sales Amount"].sum()*100:.1f}% 차지')

### 4-4. 고객 분석 — 파레토 · 구매 패턴

In [ ]:
# ── 고객 구매 집중도 (파레토 분석) ──────────────────────────────────────────
cust_sales = (internet_df.groupby('CustomerKey')['Sales Amount']
              .sum().sort_values(ascending=False))
cum_pct  = cust_sales.cumsum() / cust_sales.sum() * 100
cust_pct = np.arange(1, len(cust_sales)+1) / len(cust_sales) * 100

fig, ax1 = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#0A1628')

ax1.fill_between(cust_pct, cust_sales.values/1e3, alpha=0.25, color=PALETTE[0])
ax1.plot(cust_pct, cust_sales.values/1e3, color=PALETTE[0], linewidth=1.5)
ax1.set_xlabel('고객 누적 비율 (%)')
ax1.set_ylabel('개인 구매액 ($K)', color=PALETTE[0])
ax1.tick_params(axis='y', labelcolor=PALETTE[0])

ax2 = ax1.twinx()
ax2.plot(cust_pct, cum_pct.values, color=PALETTE[1], linewidth=2.5, linestyle='--')
ax2.set_ylabel('누적 매출 비율 (%)', color=PALETTE[1])
ax2.tick_params(axis='y', labelcolor=PALETTE[1])
ax2.axhline(80, color=PALETTE[3], linestyle=':', linewidth=1.2)
ax2.set_facecolor('#0A1628')

# 80% 기준선
idx_80 = np.searchsorted(cum_pct.values, 80)
ax1.axvline(cust_pct[idx_80], color=PALETTE[3], linestyle=':', linewidth=1.2)
ax1.text(cust_pct[idx_80]+1, cust_sales.iloc[0]/1e3*0.6,
         f'상위 {cust_pct[idx_80]:.0f}% = 매출 80%',
         color=PALETTE[3], fontsize=9)
ax1.set_title('고객 구매 집중도 — 파레토 분석', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

segs = {10:0.1, 25:0.25}
for pct, frac in segs.items():
    contrib = cust_sales.nlargest(int(len(cust_sales)*frac)).sum() / cust_sales.sum() * 100
    print(f'📌 상위 {pct:2d}% 고객 매출 기여 → {contrib:.1f}%')

### 4-5. 상품 분석 — 카테고리 · 서브카테고리

In [ ]:
# ── 카테고리 도넛 + 서브카테고리 Top10 ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0A1628')

# 카테고리 도넛
cat_s = df.groupby('Category')['Sales Amount'].sum().sort_values(ascending=False)
wedges, texts, autotexts = axes[0].pie(
    cat_s.values, labels=cat_s.index, autopct='%1.1f%%',
    colors=PALETTE, startangle=90, wedgeprops=dict(width=0.55),
    pctdistance=0.75, textprops={'color':'white','fontsize':10})
for at in autotexts: at.set_fontweight('bold')
axes[0].set_title('카테고리별 매출 비중', fontsize=12, pad=10)

# 서브카테고리 Top10
subcat_s = (df.groupby('Subcategory')['Sales Amount']
            .sum().nlargest(10).sort_values(ascending=True))
colors_s  = [PALETTE[0] if v >= subcat_s.quantile(0.8) else PALETTE[2]
             for v in subcat_s.values]
axes[1].barh(subcat_s.index, subcat_s.values/1e6, color=colors_s, edgecolor='none', height=0.6)
for val, y in zip(subcat_s.values, range(len(subcat_s))):
    axes[1].text(val/1e6+0.2, y, f'${val/1e6:.1f}M', va='center',
                 fontsize=9, color='white')
axes[1].set_title('서브카테고리 매출 Top 10', fontsize=12, pad=10)
axes[1].set_xlabel('매출 ($M)')
axes[1].grid(axis='x', alpha=0.3)
axes[1].set_axisbelow(True)

plt.suptitle('상품 분석 — 카테고리 · 서브카테고리', fontsize=14,
             fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.show()

print('📌 특이현상: Bikes 카테고리 건당 주문량은 평균 1.3개 → 고단가 소품목')
print('📌 특이현상: Accessories 건당 주문량 평균 7.4개 → 저단가 다품목')

### 4-6. 고객 × 상품 교차 분석 (CRM)

In [ ]:
# ── 카테고리 × 채널 + 카테고리별 평균 단가 ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0A1628')

# 카테고리 × 채널
cat_ch = (df.groupby(['Category','Channel'])['Sales Amount']
          .sum().div(1e6).unstack().fillna(0))
x = np.arange(len(cat_ch)); w = 0.38
for i, (ch, color) in enumerate(zip(cat_ch.columns, [PALETTE[0], PALETTE[2]])):
    axes[0].bar(x+(i-0.5)*w, cat_ch[ch], width=w, color=color,
                label=ch, edgecolor='none')
axes[0].set_xticks(x)
axes[0].set_xticklabels(cat_ch.index, fontsize=11)
axes[0].set_ylabel('매출 ($M)')
axes[0].set_title('카테고리 × 채널별 매출')
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)
axes[0].set_axisbelow(True)

# 카테고리별 평균 단가 vs 평균 주문량 (버블 차트)
cat_stats = df.groupby('Category').agg(
    avg_price=('Unit Price','mean'),
    avg_qty  =('Order Quantity','mean'),
    total_s  =('Sales Amount','sum')
).reset_index()
sizes = (cat_stats['total_s'] / cat_stats['total_s'].max() * 1500).values
scatter = axes[1].scatter(cat_stats['avg_price'], cat_stats['avg_qty'],
                          s=sizes, c=PALETTE[:len(cat_stats)],
                          alpha=0.8, edgecolors='white', linewidths=0.8)
for _, row in cat_stats.iterrows():
    axes[1].annotate(row['Category'],
                     (row['avg_price'], row['avg_qty']),
                     textcoords='offset points', xytext=(8,4),
                     fontsize=9, color='white')
axes[1].set_xlabel('평균 단가 ($)')
axes[1].set_ylabel('평균 주문 수량')
axes[1].set_title('카테고리별 단가 vs 주문량 (버블=총매출)')
axes[1].grid(alpha=0.3)

plt.suptitle('고객 × 상품 교차 분석', fontsize=14,
             fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.show()

## 5️⃣ 예측 모델 (Random Forest)
### 5-1. 피처 엔지니어링

In [ ]:
# ── LabelEncoding ────────────────────────────────────────────────────────────
le = {}
ENCODE_COLS = ['Subcategory','Region','Channel','CustCountry','Category']

for col in ENCODE_COLS:
    le[col] = LabelEncoder()
    df[col+'_enc'] = le[col].fit_transform(df[col].fillna('Unknown'))

print('✅ 인코딩 완료')
for col in ENCODE_COLS:
    print(f'  {col}: {list(le[col].classes_)}')

### 5-2. Classification — Bikes Buy or Not Buy

In [ ]:
# ── 고객 레벨 집계 ───────────────────────────────────────────────────────────
le_ctry = LabelEncoder()

cust_feat = (
    internet_df.groupby('CustomerKey').agg(
        avg_spend    = ('Sales Amount',          lambda x: x.sum()/len(x)),
        total_qty    = ('Order Quantity',         'sum'),
        num_orders   = ('SalesOrderLineKey',      'count'),
        avg_disc     = ('Unit Price Discount Pct','mean'),
        country      = ('CustCountry',            lambda x: x.mode()[0]),
        bought_bikes = ('Category',               lambda x: int('Bikes' in x.values))
    ).reset_index()
)
cust_feat['country_enc'] = le_ctry.fit_transform(cust_feat['country'].fillna('Unknown'))

FEATURES_CLS = ['avg_spend','total_qty','num_orders','avg_disc','country_enc']
X_c = cust_feat[FEATURES_CLS]
y_c = cust_feat['bought_bikes']

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
clf.fit(X_tr_c, y_tr_c)
y_pred_c = clf.predict(X_te_c)

print(f'Train: {len(X_tr_c):,}건  |  Test: {len(X_te_c):,}건')
print(f'Bikes 구매자 비율: {y_c.mean()*100:.1f}%\n')
print(classification_report(y_te_c, y_pred_c, target_names=['Non-Bike','Bike']))

In [ ]:
# ── Classifier 시각화 ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0A1628')

ConfusionMatrixDisplay.from_predictions(
    y_te_c, y_pred_c, display_labels=['Non-Bike','Bike'],
    cmap='Blues', ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix', pad=10)
axes[0].set_facecolor('#0D2240')

fi_c = pd.Series(clf.feature_importances_, index=FEATURES_CLS).sort_values(ascending=True)
colors_fi = [PALETTE[0] if v == fi_c.max() else PALETTE[2] for v in fi_c.values]
fi_c.plot(kind='barh', ax=axes[1], color=colors_fi, edgecolor='none')
axes[1].set_title('Feature Importance (Classifier)', pad=10)
axes[1].set_xlabel('중요도')
axes[1].grid(axis='x', alpha=0.3)
axes[1].set_axisbelow(True)

plt.suptitle('RF Classifier — Bikes Buy or Not Buy',
             fontsize=13, fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.show()

### 5-3. Regression — 주문 수량 예측

In [ ]:
# ── Regressor 학습 ───────────────────────────────────────────────────────────
FEATURES_REG = ['Category_enc','Subcategory_enc','Region_enc','Channel_enc','month_num']
X_r = df[FEATURES_REG]
y_r = df['Order Quantity']

X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_r, y_r, test_size=0.2, random_state=42)

reg = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)
reg.fit(X_tr_r, y_tr_r)
y_pred_r = reg.predict(X_te_r)

mae = mean_absolute_error(y_te_r, y_pred_r)
r2  = r2_score(y_te_r, y_pred_r)

print(f'Train: {len(X_tr_r):,}건  |  Test: {len(X_te_r):,}건')
print(f'MAE    : {mae:.3f} 개  (평균 {mae:.2f}개 오차)')
print(f'R² Score: {r2:.3f}')

In [ ]:
# ── Regressor 시각화 ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0A1628')

# Actual vs Predicted
np.random.seed(42)
idx = np.random.choice(len(y_te_r), 500, replace=False)
act  = np.array(y_te_r)[idx]
pred = y_pred_r[idx]
axes[0].scatter(act, pred, alpha=0.4, s=20, color=PALETTE[0], edgecolors='none')
mn, mx = min(act.min(), pred.min()), max(act.max(), pred.max())
axes[0].plot([mn,mx],[mn,mx], '--', color=PALETTE[3], linewidth=1.5, label='y=x')
axes[0].set_xlabel('Actual'); axes[0].set_ylabel('Predicted')
axes[0].set_title(f'Actual vs Predicted  (R²={r2:.3f})', pad=10)
axes[0].legend(); axes[0].grid(alpha=0.3)

# Feature Importance
fi_r = pd.Series(reg.feature_importances_, index=FEATURES_REG).sort_values(ascending=True)
colors_r = [PALETTE[1] if v == fi_r.max() else PALETTE[2] for v in fi_r.values]
fi_r.plot(kind='barh', ax=axes[1], color=colors_r, edgecolor='none')
axes[1].set_title('Feature Importance (Regressor)', pad=10)
axes[1].set_xlabel('중요도')
axes[1].grid(axis='x', alpha=0.3)
axes[1].set_axisbelow(True)

plt.suptitle('RF Regressor — Order Quantity 예측',
             fontsize=13, fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.show()

### 5-4. 🔮 인터랙티브 예측 시뮬레이션

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 🎯 CLASSIFICATION — 고객 정보 입력 → Bikes 구매 여부 예측
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('[ 고객 정보를 직접 입력하세요 ]')
print('  국가 선택지:', list(le_ctry.classes_))
print()

# ✏️  아래 값을 원하는 대로 바꿔보세요
INPUT_avg_spend  = 2500       # 고객 평균 주문 금액 ($)
INPUT_total_qty  = 5          # 총 구매 수량 (개)
INPUT_num_orders = 3          # 총 주문 횟수
INPUT_avg_disc   = 0.0        # 평균 할인율 (0.0 ~ 1.0)
INPUT_country    = 'United States'   # 국가

country_encoded = le_ctry.transform([INPUT_country])[0]
new_customer = pd.DataFrame([{
    'avg_spend':   INPUT_avg_spend,
    'total_qty':   INPUT_total_qty,
    'num_orders':  INPUT_num_orders,
    'avg_disc':    INPUT_avg_disc,
    'country_enc': country_encoded,
}])

prob  = clf.predict_proba(new_customer)[0]
label = clf.predict(new_customer)[0]

print('─' * 45)
print(f'  avg_spend  : ${INPUT_avg_spend:,.0f}')
print(f'  total_qty  : {INPUT_total_qty}개')
print(f'  num_orders : {INPUT_num_orders}회')
print(f'  country    : {INPUT_country}')
print('─' * 45)
print(f'  예측 결과  : {"🚴 Bikes 구매할 것!" if label==1 else "❌ Bikes 구매 안 할 것"}')
print(f'  Non-Bike 확률 : {prob[0]*100:.1f}%')
print(f'  Bike 확률     : {prob[1]*100:.1f}%')
print('─' * 45)

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 📈 REGRESSION — 상품·지역 입력 → 예상 판매 수량 예측
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print('[ 상품·지역 조건을 직접 입력하세요 ]')
print('  카테고리   :', list(le['Category'].classes_))
print('  서브카테고리:', list(le['Subcategory'].classes_))
print('  지역       :', list(le['Region'].classes_))
print('  채널       :', list(le['Channel'].classes_))
print()

# ✏️  아래 값을 원하는 대로 바꿔보세요
INPUT_category    = 'Bikes'
INPUT_subcategory = 'Road Bikes'
INPUT_region      = 'Northwest'
INPUT_channel     = 'Internet'
INPUT_month       = 5          # 월 (1~12)

new_order = pd.DataFrame([{
    'Category_enc':    le['Category'].transform([INPUT_category])[0],
    'Subcategory_enc': le['Subcategory'].transform([INPUT_subcategory])[0],
    'Region_enc':      le['Region'].transform([INPUT_region])[0],
    'Channel_enc':     le['Channel'].transform([INPUT_channel])[0],
    'month_num':       INPUT_month,
}])

qty_pred = reg.predict(new_order)[0]

print('─' * 45)
print(f'  Category    : {INPUT_category}')
print(f'  Subcategory : {INPUT_subcategory}')
print(f'  Region      : {INPUT_region}')
print(f'  Channel     : {INPUT_channel}')
print(f'  Month       : {INPUT_month}월')
print('─' * 45)
print(f'  📦 예측 주문 수량 : {qty_pred:.2f} 개')
print('─' * 45)

## 6️⃣ CRM 인사이트 요약

In [ ]:
# ── 모델 성능 + CRM 전략 요약 ────────────────────────────────────────────────
print('=' * 60)
print('  📊 모델 성능 요약')
print('=' * 60)
print(f'  [Classification]')
print(f'    Accuracy    : {accuracy_score(y_te_c, y_pred_c)*100:.1f}%')
print(f'    Top Feature : avg_spend (구매력이 Bikes 구매를 결정)')
print()
print(f'  [Regression]')
print(f'    R² Score    : {r2:.3f}')
print(f'    MAE         : {mae:.2f}개')
print(f'    Top Feature : Channel ({reg.feature_importances_[FEATURES_REG.index("Channel_enc")]*100:.0f}%)')
print()
print('=' * 60)
print('  🎯 CRM 전략 제언 (데이터 근거 기반)')
print('=' * 60)
strategies = [
    ('1. 고가치 고객 Lock-in',   '상위 10% 고객 → 매출 40% 기여 → 멤버십·신모델 우선 공개'),
    ('2. Accessories 교차판매',  'Bikes 구매 후 72h 이내 자동 이메일 → 반복 구매 유도'),
    ('3. 봄 시즌 캠페인',        '3~5월 피크 확인 → 3월 Early Bird 할인 진행'),
    ('4. US·AU 집중 투자',       '두 국가 합산 62% 매출 → 로컬 이벤트·무료 배송 프로모션'),
]
for title, desc in strategies:
    print(f'  {title}')
    print(f'  └ {desc}\n')